# **XGBoost Model Development**

I320D Final Project: Predicting Ambulance Arrival Times

---

**Part 1:** Preprocessing and feature engineering

**Part 2:** Gradient Boosted Random Forest Regression model development

In [1]:
from math import log
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
import sklearn.metrics as metrics
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr
from matplotlib import pyplot as plt
from google.colab import userdata
from google.colab import files
from scipy import sparse

## Part 1: Preprocessing & feature engineering
* Extract data
* Create new features (time, lag, rolling avg)
* Merge weather data onto EMS data
* Select target variable & features
* Scale (MinMax or Standard) & One-hot encode
* Split Data for training & testing


Extract & Clean Data

In [2]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# import EMS data
ems_df = pd.read_csv('/content/drive/MyDrive/Academics/Y3_Fa25/I320D_ML/final-project/nyc_ems_2023_2024_data.csv',
                     parse_dates=['incident_datetime', 'first_on_scene_datetime'])

In [4]:
# View df info
ems_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2529402 entries, 0 to 2529401
Data columns (total 16 columns):
 #   Column                         Dtype         
---  ------                         -----         
 0   cad_incident_id                int64         
 1   incident_datetime              datetime64[ns]
 2   initial_call_type              object        
 3   initial_severity_level_code    int64         
 4   final_call_type                object        
 5   final_severity_level_code      int64         
 6   first_assignment_datetime      object        
 7   dispatch_response_seconds_qy   int64         
 8   first_on_scene_datetime        datetime64[ns]
 9   incident_response_seconds_qy   int64         
 10  incident_travel_tm_seconds_qy  int64         
 11  incident_close_datetime        object        
 12  borough                        object        
 13  incident_dispatch_area         object        
 14  zipcode                        int64         
 15  special_event_i

In [5]:
# Drop rows where 'response_time_seconds' (the target variable) is NaN
ems_df['response_time_seconds'] = (ems_df['first_on_scene_datetime'] - ems_df['incident_datetime']).dt.total_seconds()
ems_df.dropna(subset=['response_time_seconds'], inplace=True)

Feature Engineering

In [6]:
# Define a meaningful hour colum --> incident_hour (0-23)
ems_df['incident_hour'] = ems_df['incident_datetime'].dt.hour

# Define a meaningful day column --> incident_day_of_week (0-6)
ems_df['incident_day_of_week'] = ems_df['incident_datetime'].dt.dayofweek

# Define a meaningful month column --> incident_month (1-12)
ems_df['incident_month'] = ems_df['incident_datetime'].dt.month

In [7]:
# Indicate whether an incident occurs on a weekend
ems_df['is_weekend'] = ems_df['incident_day_of_week'].isin([5, 6]).astype(int)

# Indicate whether an incident occurs during rush hour
is_weekday = ~ems_df['incident_day_of_week'].isin([5, 6])
is_morning_rush = (ems_df['incident_hour'] >= 6) & (ems_df['incident_hour'] <= 10)          # morning rush: 6-10AM
is_evening_rush = (ems_df['incident_hour'] >= 16) & (ems_df['incident_hour'] <= 20)         # evening rush: 4-8PM
ems_df['is_rush_hour'] = ((is_morning_rush | is_evening_rush) & is_weekday).astype(int) # binary categorical

# Turn special_event_indicator into a binary numerical categorical variable
ems_df['special_event_indicator'] = ems_df['special_event_indicator'].map({'Y': 1, 'N': 0}).astype(int)

Lag & Rolling Features

In [8]:
# Create a lag feature: response time of previous incident in the same service area
ems_df['lag_response_time'] = ems_df.groupby('incident_dispatch_area')['response_time_seconds'].shift(1)

# For rows that don't have a previous response time, impute with overall mean response time
avg_response_time = ems_df['response_time_seconds'].mean()
ems_df['lag_response_time'] = ems_df['lag_response_time'].fillna(avg_response_time)

In [9]:
# Ensure chronological order before calculating rolling avg
ems_df = ems_df.sort_values(by='incident_datetime').reset_index(drop=True)

# Create a rolling avg. response time feature for the past 3 hours
ems_df['rolling_avg_response_time_3h'] = (ems_df
    .groupby('incident_dispatch_area', observed=True)            # group by dispatch area
    [['incident_datetime', 'response_time_seconds']]             # select both columns
    .rolling(window='3h', on='incident_datetime', closed='left') # 3hr window <-- we can change this to a different value too
    .mean()                                                      # calculate avg
    ['response_time_seconds'] # use the response_time_seconds column from the result
    .reset_index(level=0, drop=True)
)

# For rows that don't have 3hr of previous data, impute with overall mean response time
ems_df['rolling_avg_response_time_3h'] = ems_df['rolling_avg_response_time_3h'].fillna(avg_response_time)

In [10]:
# Create a lagged incident count feature for the past 15 minutes
ems_df['lag_incident_count_15min'] = (ems_df
    .groupby('incident_dispatch_area', observed=True)               # group by dispatch are
    .rolling(window='15min', on='incident_datetime', closed='left') # 15min window <-- we can change this to a different value too
    .count()                                                        # calculate total count
    ['cad_incident_id'] # use the count for cad_incident_id (ensures integer type)
    .reset_index(level=0, drop=True)
)

# For rows that don't have any incidents within the last 15min - impute NaN with 0 and ensure integer type
ems_df['lag_incident_count_15min'] = ems_df['lag_incident_count_15min'].fillna(0).astype(int)

In [11]:
ems_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2529402 entries, 0 to 2529401
Data columns (total 25 columns):
 #   Column                         Dtype         
---  ------                         -----         
 0   cad_incident_id                int64         
 1   incident_datetime              datetime64[ns]
 2   initial_call_type              object        
 3   initial_severity_level_code    int64         
 4   final_call_type                object        
 5   final_severity_level_code      int64         
 6   first_assignment_datetime      object        
 7   dispatch_response_seconds_qy   int64         
 8   first_on_scene_datetime        datetime64[ns]
 9   incident_response_seconds_qy   int64         
 10  incident_travel_tm_seconds_qy  int64         
 11  incident_close_datetime        object        
 12  borough                        object        
 13  incident_dispatch_area         object        
 14  zipcode                        int64         
 15  special_event_i

In [12]:
ems_df.head()

,cad_incident_id,incident_datetime,initial_call_type,initial_severity_level_code,final_call_type,final_severity_level_code,first_assignment_datetime,dispatch_response_seconds_qy,first_on_scene_datetime,incident_response_seconds_qy,...,special_event_indicator,response_time_seconds,incident_hour,incident_day_of_week,incident_month,is_weekend,is_rush_hour,lag_response_time,rolling_avg_response_time_3h,lag_incident_count_15min
0,230010003,2023-01-01 00:00:30,DIFFFC,2,DIFFFC,2,2023-01-01 00:02:24,114,2023-01-01 00:08:29,479,...,0,479.0,0,6,1,1,0,548.336523,548.336523,0
1,230010004,2023-01-01 00:01:35,INJURY,5,INJURY,5,2023-01-01 01:03:40,3725,2023-01-01 01:03:40,3725,...,0,3725.0,0,6,1,1,0,548.336523,548.336523,0
2,230010011,2023-01-01 00:03:22,UNC,2,UNC,2,2023-01-01 00:08:20,298,2023-01-01 00:21:37,1095,...,0,1095.0,0,6,1,1,0,548.336523,548.336523,0
3,230010012,2023-01-01 00:03:37,UNC,2,DRUG,4,2023-01-01 00:23:48,1211,2023-01-01 00:46:52,2595,...,0,2595.0,0,6,1,1,0,1095.000000,1095.000000,1
4,230010013,2023-01-01 00:03:53,SICK,6,SICK,6,2023-01-01 00:04:17,24,2023-01-01 00:14:46,653,...,0,653.0,0,6,1,1,0,548.336523,548.336523,0


#### Merge weather data

In [13]:
# load ERA5 weather data
weather_df = pd.read_csv('/content/drive/MyDrive/Academics/Y3_Fa25/I320D_ML/final-project/era5_2023_2024_weather_data.csv')

In [14]:
# Convert index to datetime type
weather_df['Unnamed: 0'] = pd.to_datetime(weather_df['Unnamed: 0'])
weather_df = weather_df.set_index('Unnamed: 0')
weather_df.index.name = 'datetime'
weather_df.index.dtype

dtype('<M8[ns]')

In [15]:
# Make temporary column to find time of EMS incident to the nearest hour
ems_df['merge_hour'] = ems_df['incident_datetime'].dt.floor('h')

# Merge weather df onto EMS df
df_merged = pd.merge(
    ems_df,
    weather_df,
    left_on='merge_hour',
    right_index=True,
    how='left'
)

# Drop the temporary merge column
df_merged = df_merged.drop(columns=['merge_hour'])
df_merged.head()

,cad_incident_id,incident_datetime,initial_call_type,initial_severity_level_code,final_call_type,final_severity_level_code,first_assignment_datetime,dispatch_response_seconds_qy,first_on_scene_datetime,incident_response_seconds_qy,...,lag_response_time,rolling_avg_response_time_3h,lag_incident_count_15min,snow_depth,temp_2m,snowfall,total_precipitation,avg_total_snowfall_rate,avg_total_precipitation_rate,precipitation_type
0,230010003,2023-01-01 00:00:30,DIFFFC,2,DIFFFC,2,2023-01-01 00:02:24,114,2023-01-01 00:08:29,479,...,548.336523,548.336523,0,0.0,283.28027,NaN,NaN,NaN,NaN,NaN
0,230010003,2023-01-01 00:00:30,DIFFFC,2,DIFFFC,2,2023-01-01 00:02:24,114,2023-01-01 00:08:29,479,...,548.336523,548.336523,0,NaN,NaN,0.0,0.000723,0.0,0.000201,1.0
1,230010004,2023-01-01 00:01:35,INJURY,5,INJURY,5,2023-01-01 01:03:40,3725,2023-01-01 01:03:40,3725,...,548.336523,548.336523,0,0.0,283.28027,NaN,NaN,NaN,NaN,NaN
1,230010004,2023-01-01 00:01:35,INJURY,5,INJURY,5,2023-01-01 01:03:40,3725,2023-01-01 01:03:40,3725,...,548.336523,548.336523,0,NaN,NaN,0.0,0.000723,0.0,0.000201,1.0
2,230010011,2023-01-01 00:03:22,UNC,2,UNC,2,2023-01-01 00:08:20,298,2023-01-01 00:21:37,1095,...,548.336523,548.336523,0,0.0,283.28027,NaN,NaN,NaN,NaN,NaN


In [16]:
# Initial merge outputs repeated rows
# Consolidate weather_df to have a single row per hour by selecting non-null values
weather_df_cleaned = weather_df.groupby(level=0).max() # group by datetime index
display(weather_df_cleaned.head())
weather_df_cleaned.info()

,snow_depth,temp_2m,snowfall,total_precipitation,avg_total_snowfall_rate,avg_total_precipitation_rate,precipitation_type
datetime,,,,,,,
2022-12-31 19:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-12-31 20:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-12-31 21:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-12-31 22:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-12-31 23:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 17556 entries, 2022-12-31 19:00:00 to 2025-01-01 06:00:00
Data columns (total 7 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   snow_depth                    17544 non-null  float64
 1   temp_2m                       17544 non-null  float64
 2   snowfall                      17544 non-null  float64
 3   total_precipitation           17544 non-null  float64
 4   avg_total_snowfall_rate       17544 non-null  float64
 5   avg_total_precipitation_rate  17544 non-null  float64
 6   precipitation_type            17544 non-null  float64
dtypes: float64(7)
memory usage: 1.1 MB


In [17]:
# Identify weather-related columns that have NaNs after the merge
weather_columns_with_nans = [
    'snow_depth',
    'temp_2m',
    'snowfall',
    'total_precipitation',
    'avg_total_snowfall_rate',
    'avg_total_precipitation_rate',
    'precipitation_type'
]

# Apply forward-fill to these columns in df_merged
df_merged = df_merged.sort_values(by='incident_datetime').reset_index(drop=True)
df_merged[weather_columns_with_nans] = df_merged[weather_columns_with_nans].ffill()

In [18]:
# Merge weather data onto ems data using cleaned weather_df

# Temporary column to find time of EMS incident to the nearest hour
ems_df['merge_hour'] = ems_df['incident_datetime'].dt.floor('h')

# Merge cleaned weather_df onto EMS df
df_merged = pd.merge(
    ems_df,
    weather_df_cleaned,
    left_on='merge_hour',
    right_index=True,
    how='left'
)

# Drop the temporary merge column
df_merged = df_merged.drop(columns=['merge_hour'])

In [19]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2529402 entries, 0 to 2529401
Data columns (total 32 columns):
 #   Column                         Dtype         
---  ------                         -----         
 0   cad_incident_id                int64         
 1   incident_datetime              datetime64[ns]
 2   initial_call_type              object        
 3   initial_severity_level_code    int64         
 4   final_call_type                object        
 5   final_severity_level_code      int64         
 6   first_assignment_datetime      object        
 7   dispatch_response_seconds_qy   int64         
 8   first_on_scene_datetime        datetime64[ns]
 9   incident_response_seconds_qy   int64         
 10  incident_travel_tm_seconds_qy  int64         
 11  incident_close_datetime        object        
 12  borough                        object        
 13  incident_dispatch_area         object        
 14  zipcode                        int64         
 15  special_event_i

In [20]:
df_merged.head()

,cad_incident_id,incident_datetime,initial_call_type,initial_severity_level_code,final_call_type,final_severity_level_code,first_assignment_datetime,dispatch_response_seconds_qy,first_on_scene_datetime,incident_response_seconds_qy,...,lag_response_time,rolling_avg_response_time_3h,lag_incident_count_15min,snow_depth,temp_2m,snowfall,total_precipitation,avg_total_snowfall_rate,avg_total_precipitation_rate,precipitation_type
0,230010003,2023-01-01 00:00:30,DIFFFC,2,DIFFFC,2,2023-01-01 00:02:24,114,2023-01-01 00:08:29,479,...,548.336523,548.336523,0,0.0,283.28027,0.0,0.000723,0.0,0.000201,1.0
1,230010004,2023-01-01 00:01:35,INJURY,5,INJURY,5,2023-01-01 01:03:40,3725,2023-01-01 01:03:40,3725,...,548.336523,548.336523,0,0.0,283.28027,0.0,0.000723,0.0,0.000201,1.0
2,230010011,2023-01-01 00:03:22,UNC,2,UNC,2,2023-01-01 00:08:20,298,2023-01-01 00:21:37,1095,...,548.336523,548.336523,0,0.0,283.28027,0.0,0.000723,0.0,0.000201,1.0
3,230010012,2023-01-01 00:03:37,UNC,2,DRUG,4,2023-01-01 00:23:48,1211,2023-01-01 00:46:52,2595,...,1095.000000,1095.000000,1,0.0,283.28027,0.0,0.000723,0.0,0.000201,1.0
4,230010013,2023-01-01 00:03:53,SICK,6,SICK,6,2023-01-01 00:04:17,24,2023-01-01 00:14:46,653,...,548.336523,548.336523,0,0.0,283.28027,0.0,0.000723,0.0,0.000201,1.0


In [21]:
df_merged.tail()

,cad_incident_id,incident_datetime,initial_call_type,initial_severity_level_code,final_call_type,final_severity_level_code,first_assignment_datetime,dispatch_response_seconds_qy,first_on_scene_datetime,incident_response_seconds_qy,...,lag_response_time,rolling_avg_response_time_3h,lag_incident_count_15min,snow_depth,temp_2m,snowfall,total_precipitation,avg_total_snowfall_rate,avg_total_precipitation_rate,precipitation_type
2529397,243665316,2024-12-31 23:56:43,INJURY,5,INJURY,5,2024-12-31 23:56:50,7,2025-01-01 00:04:51,488,...,654.0,782.714286,0,0.0,282.20963,0.0,0.0,0.0,0.0,0.0
2529398,243665317,2024-12-31 23:57:42,DRUG,5,DRUG,5,2024-12-31 23:58:30,48,2025-01-01 00:08:17,635,...,819.0,647.111111,2,0.0,282.20963,0.0,0.0,0.0,0.0,0.0
2529399,243665321,2024-12-31 23:58:19,DIFFBR,2,DIFFBR,2,2024-12-31 23:58:31,12,2025-01-01 00:03:51,332,...,742.0,489.266667,1,0.0,282.20963,0.0,0.0,0.0,0.0,0.0
2529400,243665322,2024-12-31 23:58:33,UNC,2,UNC,2,2024-12-31 23:59:04,31,2025-01-01 00:07:25,532,...,488.0,745.875000,1,0.0,282.20963,0.0,0.0,0.0,0.0,0.0
2529401,243665324,2024-12-31 23:59:48,ABDPN,5,ABDPN,5,2025-01-01 00:13:40,832,2025-01-01 00:14:31,883,...,935.0,540.825000,3,0.0,282.20963,0.0,0.0,0.0,0.0,0.0


Select target variable and features; one-hot encode categorical features

In [22]:
df = df_merged

In [23]:
# Define target variable
target_variable_df = df[['incident_travel_tm_seconds_qy']]

# Define numerical and categorical features
numerical_features = [
    'initial_severity_level_code',
    'final_severity_level_code',
    'lag_response_time',
    'rolling_avg_response_time_3h',
    'lag_incident_count_15min',
    #'snow_depth',
    'temp_2m',
    'snowfall',
    'total_precipitation',
    'avg_total_snowfall_rate',
    'avg_total_precipitation_rate',
    #'precipitation_type'
]

# Binary numeric features (no one-hot encoding needed)
binary_numeric_features = [
    'special_event_indicator',
    'is_weekend',
    'is_rush_hour'
]

# Features we need to one-hot encode
categorical_features = [
    'borough',
    'incident_dispatch_area',
    'incident_hour',
    'incident_day_of_week',
    'incident_month'
]

In [24]:
# Combine all selected features
all_features = numerical_features + binary_numeric_features + categorical_features
features_df = df[all_features].copy()

# Update features_columns and target_variable_columns for clarity
features_columns = all_features
target_variable_columns = list(target_variable_df.columns)

Split, scale, encode data

In [25]:
# Split the data into 80% train and 20% "holder data"
x_train_raw, x_main_raw, y_train, y_main = train_test_split(features_df, target_variable_df, test_size=0.20, random_state=42)

# Split the "holder data" into 10% for test and 10% validation
x_val_raw, x_test_raw, y_val, y_test = train_test_split(x_main_raw, y_main, test_size=0.50, random_state=42)

In [26]:
# Manual Scaling for Numerical Features (using raw data as there are no NaNs)
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train_raw[numerical_features])
x_val_scaled = scaler.transform(x_val_raw[numerical_features])
x_test_scaled = scaler.transform(x_test_raw[numerical_features])

In [27]:
# Manual One-Hot Encoding for Categorical Features
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output = True)
x_train_encoded = encoder.fit_transform(x_train_raw[categorical_features])
x_val_encoded = encoder.transform(x_val_raw[categorical_features])
x_test_encoded = encoder.transform(x_test_raw[categorical_features])

In [28]:
# Concatenate processed numerical and categorical features
x_train_processed = np.hstack((x_train_scaled, x_train_encoded.toarray()))
x_val_processed = np.hstack((x_val_scaled, x_val_encoded.toarray()))
x_test_processed = np.hstack((x_test_scaled, x_test_encoded.toarray()))

# Printing of shapes for validation
print (f"Training: Features' shape =  {x_train_processed.shape}")
print (f"Training: Label's shape = {y_train.shape}")

print (f"Validation: Features' shape =  {x_val_processed.shape}")
print (f"Validation: Label's shape = {y_val.shape}")

print (f"Test: Features' shape =  {x_test_processed.shape}")
print (f"Test: Label's shape = {y_test.shape}")

Training: Features' shape =  (2023521, 93)
Training: Label's shape = (2023521, 1)
Validation: Features' shape =  (252940, 93)
Validation: Label's shape = (252940, 1)
Test: Features' shape =  (252941, 93)
Test: Label's shape = (252941, 1)


## **Part 2: Gradient Boosted Random Forest Regressor**

**Steps:**
1. Feature Engineering
2. Feature Selection & Data Splitting
3. Define, Train, and Test Model
4. Evaluate Model

In [32]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, r2_score

In [33]:
# Define GBRT (gradient boosted random forest regression) model
gbrt_model = xgb.XGBRegressor(
    n_estimators = 500,
    max_depth = 7,
    learning_rate = 0.05,
    random_state = 42,
    # n_jobs = -1
)

# Train model
gbrt_model.fit(x_train_processed, y_train.to_numpy().flatten())

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

In [34]:
# Test model
gbrt_y_pred = gbrt_model.predict(x_test_processed)

# Print accuracy metrics
gbrt_MSE = metrics.mean_squared_error(y_test, gbrt_y_pred)
gbrt_RMSE = np.sqrt(gbrt_MSE)
gbrt_MAE = metrics.mean_absolute_error(y_test, gbrt_y_pred)
gbrt_MedAE = metrics.median_absolute_error(y_test, gbrt_y_pred)
gbrt_R2 = r2_score(y_test, gbrt_y_pred)

print('GBRT Model Results\n---')
print (f"Mean Squared Error (MSE) = {gbrt_MSE:.3f}")
print (f"Root Mean Squared Error (RMSE)= {gbrt_RMSE:.3f}") # error in the original units (sec)
print('---')
print (f"Mean Absolute Error (MAE)= {gbrt_MAE:.3f}")
print (f"Median Absolute Error (MedAE)= {gbrt_MedAE:.3f}") # more resilient to outliers
print (f"R-squared Score (R2) = {gbrt_R2:.3f}")

GBRT Model Results
---
Mean Squared Error (MSE) = 95241.531
Root Mean Squared Error (RMSE)= 308.612
---
Mean Absolute Error (MAE)= 209.729
Median Absolute Error (MedAE)= 153.830
R-squared Score (R2) = 0.175
